In [1]:
import json, torch, re
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML

MODEL_PATH = "/home/lsuryana/.cache/huggingface/hub/Qwen2-VL-7B-Instruct"
print("Loading model...")
model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_PATH, torch_dtype="auto", device_map="cuda")
processor = AutoProcessor.from_pretrained(MODEL_PATH)
print(f"Model loaded. VRAM used: {round(torch.cuda.memory_allocated()/1e9, 2)} GB")

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading model...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded. VRAM used: 16.63 GB


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: Load data and define helper functions
# ─────────────────────────────────────────────────────────────────────────────

# preprocessed.json is the output of scripts/02_preprocess.py
# It contains one entry per selected sample with the following fields:
#
#   sample_token  : unique ID of this keyframe in nuScenes
#   scene_token   : unique ID of the scene this keyframe belongs to
#   question      : the planning question from DriveLM (used as VLM prompt)
#   future_tokens : list of 3 nuScenes sample tokens for t+0.5s, t+1.0s, t+1.5s
#   frame_dir     : path to the folder containing t0.jpg, t1.jpg, t2.jpg, t3.jpg
#   trajectory    : list of 3 dicts {x, y} = ego vehicle position at each future
#                   timestep, in vehicle-relative coordinates
#                   x = meters forward (positive = moving forward)
#                   y = meters sideways (positive = left, negative = right)
#
# Example entry:
# {
#   "sample_token": "c5f58c19...",
#   "scene_token": "916...",
#   "question": "What actions could the ego vehicle take?",
#   "future_tokens": ["abc...", "def...", "ghi..."],
#   "frame_dir": "cache/frames/c5f58c19...",
#   "trajectory": [{"x": 4.181, "y": -0.073},
#                  {"x": 8.351, "y": -0.257},
#                  {"x": 12.615, "y": -0.568}]
# }

with open('results/preprocessed.json') as f:
    data = json.load(f)

# ─────────────────────────────────────────────────────────────────────────────
# derive_gt_action: converts ego trajectory into a single GT action label
# Rules applied in priority order (first match wins):
#   STOP             : vehicle barely moved (x at t+1.5s < 0.5m)
#   CHANGE_LANE_LEFT : strong leftward drift (y at t+1.5s > 3.0m)
#   CHANGE_LANE_RIGHT: strong rightward drift (y at t+1.5s < -3.0m)
#   DECELERATE       : slow speed (x at t+0.5s < 1.5m, ~10 km/h)
#   ACCELERATE       : high speed (x at t+1.5s > 12.0m, ~29 km/h)
#   KEEP_LANE        : everything else
# NOTE: these thresholds are heuristics — documented as Limitation L1
# ─────────────────────────────────────────────────────────────────────────────
def derive_gt_action(traj):
    x1 = traj[0]['x']   # forward distance at t+0.5s
    x3 = traj[2]['x']   # forward distance at t+1.5s
    y3 = traj[2]['y']   # lateral distance at t+1.5s
    if abs(x3) < 0.5:   return 'STOP'
    if y3 > 3.0:        return 'CHANGE_LANE_LEFT'
    if y3 < -3.0:       return 'CHANGE_LANE_RIGHT'
    if x1 < 1.5:        return 'DECELERATE'
    if x3 > 12.0:       return 'ACCELERATE'
    return 'KEEP_LANE'

# ─────────────────────────────────────────────────────────────────────────────
# run_inference: sends a message (with images) to Qwen2-VL and returns raw text
# temperature=0.0 + do_sample=False = deterministic output (same input → same output)
# max_new_tokens=512 = maximum length of the model's response
# ─────────────────────────────────────────────────────────────────────────────
def run_inference(messages):
    # Free any leftover GPU memory before inference
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
    trimmed = out[:, inputs['input_ids'].shape[1]:]

    # Free VRAM immediately after inference
    del inputs, out, trimmed
    torch.cuda.empty_cache()
    
    return result

# ─────────────────────────────────────────────────────────────────────────────
# parse_output: extracts ANSWER, REASONING, ACTION fields from raw model output
# The model is prompted to follow a strict format — this function extracts each part
# If ACTION is not one of the 6 valid options, it is flagged as PARSE_ERROR
# ─────────────────────────────────────────────────────────────────────────────
VALID_ACTIONS = {'KEEP_LANE', 'ACCELERATE', 'DECELERATE',
                 'CHANGE_LANE_LEFT', 'CHANGE_LANE_RIGHT', 'STOP'}

def parse_output(raw):
    answer   = re.search(r'ANSWER:\s*(.*?)(?=REASONING:|$)', raw, re.DOTALL)
    reasoning = re.search(r'REASONING:\s*(.*?)(?=ACTION:|$)', raw, re.DOTALL)
    action   = re.search(r'ACTION:\s*(\w+)', raw)
    parsed_action = action.group(1).strip().upper() if action else 'PARSE_ERROR'
    if parsed_action not in VALID_ACTIONS:
        parsed_action = 'PARSE_ERROR'
    return {
        'answer':    answer.group(1).strip() if answer else '',
        'reasoning': reasoning.group(1).strip() if reasoning else '',
        'action':    parsed_action,
        'raw':       raw
    }

print(f"Loaded {len(data)} samples. Helper functions ready.")

Loaded 24 samples. Helper functions ready.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3: Smoke test — run inference on ONE sample and display results
# We use sample index 7 (sample 8) which is the first moving sample
# (samples 0-6 are from scene-0553 where the vehicle is stationary)
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_IDX = 7  # change this to view other samples (0-23)

s = data[SAMPLE_IDX]
fd   = s['frame_dir']    # folder with t0.jpg, t1.jpg, t2.jpg, t3.jpg
q    = s['question']     # planning question from DriveLM
traj = s['trajectory']   # GT ego trajectory [{x,y}, {x,y}, {x,y}]
gt_action = derive_gt_action(traj)

print(f"Sample index : {SAMPLE_IDX}")
print(f"Sample token : {s['sample_token'][:12]}...")
print(f"Question     : {q}")
print(f"Trajectory   : {traj}")
print(f"GT action    : {gt_action}")
print()

# ── Build NON POST-HOC input ──────────────────────────────────────────────────
# Input: only current frame (t0) + question
# The model has NO knowledge of what happens next
non_tmpl = open('prompts/non_posthoc.txt').read().replace('{QUESTION}', q)
msg_non = [{"role": "user", "content": [
    {"type": "image", "image": f"{fd}/t0.jpg"},
    {"type": "text",  "text": non_tmpl},
]}]

# ── Build POST-HOC input ──────────────────────────────────────────────────────
# Input: current frame (t0) + 3 future frames (t1,t2,t3) + GT trajectory
# The model CAN see what actually happened after the decision moment
pt = open('prompts/posthoc.txt').read().replace('{QUESTION}', q)
for j, wp in enumerate(traj):
    pt = pt.replace(f'{{X{j+1}}}', str(wp['x'])).replace(f'{{Y{j+1}}}', str(wp['y']))
msg_post = [{"role": "user", "content": [
    {"type": "image", "image": f"{fd}/t0.jpg"},
    {"type": "image", "image": f"{fd}/t1.jpg"},
    {"type": "image", "image": f"{fd}/t2.jpg"},
    {"type": "image", "image": f"{fd}/t3.jpg"},
    {"type": "text",  "text": pt},
]}]

# ── Run inference ─────────────────────────────────────────────────────────────
print("Running NON POST-HOC inference...")
raw_non  = run_inference(msg_non)
out_non  = parse_output(raw_non)
print("Done.")

print("Running POST-HOC inference...")
raw_post = run_inference(msg_post)
out_post = parse_output(raw_post)
print("Done.")

# ── Display images ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
titles = ['t0 (current)', 't1 (+0.5s)', 't2 (+1.0s)', 't3 (+1.5s)']
for i, (ax, title) in enumerate(zip(axes, titles)):
    img = Image.open(f"{fd}/t{i}.jpg")
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    if i == 0:
        ax.set_title(f't0 (current)\n← NON POST-HOC sees only this', fontsize=10, color='blue')
plt.suptitle(f"Scene frames — {s['sample_token'][:12]}...", fontsize=12)
plt.tight_layout()
plt.show()

# ── Display results side by side ──────────────────────────────────────────────
display(HTML(f"""
<table style="width:100%; border-collapse:collapse; font-family:monospace; font-size:13px;">
<tr style="background:#f0f0f0;">
    <th style="padding:10px; width:50%; border:1px solid #ccc;">NON POST-HOC (current frame only)</th>
    <th style="padding:10px; width:50%; border:1px solid #ccc;">POST-HOC (current + future frames + trajectory)</th>
</tr>
<tr>
    <td style="padding:10px; border:1px solid #ccc; vertical-align:top;">
        <b>ANSWER:</b><br>{out_non['answer']}<br><br>
        <b>REASONING:</b><br>{out_non['reasoning']}<br><br>
        <b>ACTION:</b> <span style="color:{'green' if out_non['action']==gt_action else 'red'}; font-weight:bold;">{out_non['action']}</span>
    </td>
    <td style="padding:10px; border:1px solid #ccc; vertical-align:top;">
        <b>ANSWER:</b><br>{out_post['answer']}<br><br>
        <b>REASONING:</b><br>{out_post['reasoning']}<br><br>
        <b>ACTION:</b> <span style="color:{'green' if out_post['action']==gt_action else 'red'}; font-weight:bold;">{out_post['action']}</span>
    </td>
</tr>
<tr style="background:#f9f9f9;">
    <td colspan="2" style="padding:10px; border:1px solid #ccc; text-align:center;">
        <b>GT ACTION:</b> {gt_action} &nbsp;|&nbsp;
        <b>Action changed?</b> {'YES ⚠' if out_non['action'] != out_post['action'] else 'NO ✓'}
    </td>
</tr>
</table>
"""))

Sample index : 7
Sample token : c5f58c19249d...
Question     : What actions could the ego vehicle take based on <c1,CAM_FRONT_RIGHT,387.5,559.2>? Why take this action and what's the probability?
Trajectory   : [{'x': 4.181, 'y': -0.073}, {'x': 8.351, 'y': -0.257}, {'x': 12.615, 'y': -0.568}]
GT action    : ACCELERATE

Running NON POST-HOC inference...
